# **Fase 5: MLOps y Produccion**

Estudiante: Maria Camila Navarrete Pinzón

Código: 2294353

Fecha: 13 febrero, 2026

# Notebook 12:  Inferencia en Produccion

**Objetivo**: Simular un pipeline de producción para generar predicciones

**Conceptos clave:**
- **Batch Inference**: Predicciones sobre grandes volúmenes de datos
- **Model Loading**: Cargar modelo desde MLflow Registry
- **Monitoring**: Verificar calidad de predicciones
- **Output Formats**: Guardar resultados para consumo (Parquet, CSV)

**Actividades:**
1. Cargar modelo desde Model Registry (Production)
2. Aplicar transformaciones del pipeline
3. Generar predicciones batch sobre datos nuevos
4. Monitorear y guardar resultados


## 1. Configuración de SparkSession

In [8]:
spark = (
    SparkSession.builder
    .appName("SECOP_Produccion")
    .master("local[2]")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
spark.sparkContext.setLogLevel("WARN")
print("Spark local listo ✅")



Spark local listo ✅


## 2. Reto 1: Cargar modelo en Production desde MLflow Registry

**Objetivo**: Cargar el modelo registrado en Production desde MLflow.

**Instrucciones**:

1. Configura la URI de MLflow
2. Define el nombre del modelo y el stage (Production)
3. Carga el modelo usando `mlflow.spark.load_model()`

**Pregunta**: ¿Por qué cargar desde el Registry en lugar de una ruta de archivo?

Cargar el modelo desde el model registry permite desacoplar el código de la versión específica del modelo. Esto facilita actualizaciones, rollback inmediato y control del ciclo de vida sin modificar el pipeline de producción.

**Pregunta**: ¿Qué ventajas tiene para un sistema de producción?

Permite gestionar versiones y estados (Staging, Production, Archived) sin modificar el pipeline. Esto facilita despliegues controlados, rollback rápido ante fallos, trazabilidad completa del modelo (métricas, parámetros, artefactos) y colaboración entre equipos. Además, garantiza consistencia entre entornos (desarrollo, pruebas y producción) y reduce el riesgo de errores humanos al evitar rutas locales o modelos hardcodeados, lo que hace el sistema más robusto, escalable y gobernable.


In [9]:
mlflow.set_tracking_uri("http://mlflow:5000")

model_name = "secop_prediccion_contratos"
model_uri = f"models:/{model_name}/Production"

print(f"Cargando modelo desde: {model_uri}")
production_model = mlflow.spark.load_model(model_uri)
print(f"Modelo cargado correctamente: {type(production_model)}")
print("MLflow UI: http://localhost:5000")


Cargando modelo desde: models:/secop_prediccion_contratos/Production


2026/02/14 17:10:07 INFO mlflow.spark: 'models:/secop_prediccion_contratos/Production' resolved as 'file:///opt/mlflow/mlruns/324944293645910354/de55e7ad2a54413480846b450fe1fdb2/artifacts/model'
2026/02/14 17:10:08 INFO mlflow.spark: URI 'models:/secop_prediccion_contratos/Production/sparkml' does not point to the current DFS.
2026/02/14 17:10:08 INFO mlflow.spark: File 'models:/secop_prediccion_contratos/Production/sparkml' not found on DFS. Will attempt to upload the file.


Modelo cargado correctamente: <class 'pyspark.ml.pipeline.PipelineModel'>
MLflow UI: http://localhost:5000


**¿Qué pasaría si no hay modelo en "Production"?**

El sistema fallaría al cargar el modelo. 

**¿Cómo manejarías ese error?**

En producción real se manejaría con try/except o fallback a Staging

## 3. Reto 2: Preparar datos nuevos para prediccion

**Objetivo**: Simular la llegada de datos nuevos para predicción.

**Instrucciones**:
1. Carga datos desde el parquet procesado
2. Renombra columnas para que coincidan con lo que espera el modelo
3. Simula datos "nuevos" eliminando la columna label

**Pregunta**: En un sistema real, ¿de dónde vendrían los datos nuevos?

En un sistema real, los datos nuevos podrían provenir de bases de datos transaccionales, APIs, archivos en S3 o sistemas de streaming como Kafka.
- Opciones: Base de datos, API, archivos S3, streaming, etc.

In [10]:
df_new = spark.read.parquet("/opt/spark-data/processed/secop_ml_ready.parquet") \
    .withColumnRenamed("features_pca", "features")

df_new_no_label = df_new.drop("valor_del_contrato_num")

print(f"Contratos para predecir: {df_new_no_label.count():,}")
print(f"Columnas: {df_new_no_label.columns}")


Contratos para predecir: 52,248
Columnas: ['features', 'label']


## 4. Reto 3: Generar predicciones batch con timestamp

**Objetivo**: Aplicar el modelo a todos los datos nuevos.

**Instrucciones**:
1. Usa `model.transform()` para generar predicciones
2. Agrega un timestamp de predicción
3. Examina las primeras predicciones

**Pregunta**: ¿Por qué agregar timestamp a las predicciones? ¿Qué otros metadatos serían útiles?

El timestamp permite trazabilidad, auditoría y análisis histórico del comportamiento del modelo. Otros metadatos útiles incluyen versión del modelo, fuente de datos y batch ID.

In [11]:
from pyspark.sql.functions import current_timestamp

predictions_batch = production_model.transform(df_new_no_label)

predictions_batch = predictions_batch.withColumn(
    "prediction_timestamp", current_timestamp()
)

predictions_batch.select(
    "prediction", "prediction_timestamp"
).show(10, truncate=False)


26/02/14 17:27:07 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/02/14 17:27:07 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


+---------------------+-------------------------+
|prediction           |prediction_timestamp     |
+---------------------+-------------------------+
|-1.2836750139294767E9|2026-02-14 17:26:55.30796|
|-2.980226826417466E9 |2026-02-14 17:26:55.30796|
|1.3258958939247262E8 |2026-02-14 17:26:55.30796|
|9.131281901357667E8  |2026-02-14 17:26:55.30796|
|-1.3798448166652741E9|2026-02-14 17:26:55.30796|
|-8.163048280505522E9 |2026-02-14 17:26:55.30796|
|9.30894323455531E8   |2026-02-14 17:26:55.30796|
|8.2325612828924675E9 |2026-02-14 17:26:55.30796|
|1.4418520121369042E9 |2026-02-14 17:26:55.30796|
|3.8696288137864614E9 |2026-02-14 17:26:55.30796|
+---------------------+-------------------------+
only showing top 10 rows



## 5. Reto 4: Monitorear predicciones (estadisticas, anomalias, rangos)

**Objetivo**: Verificar que las predicciones son razonables.

**Instrucciones**:
1. Calcula estadísticas básicas de las predicciones (min, max, avg, std)
2. Identifica predicciones negativas (si no tienen sentido en tu contexto)
3. Analiza la distribución por rangos
4. Define alertas para anomalías

**Pregunta**: ¿Cómo detectarías "data drift" (cambio en la distribución de datos)? ¿Qué harías si las predicciones empiezan a ser muy diferentes de lo esperado?

El data drift se detecta comparando distribuciones actuales con las de entrenamiento. Si las predicciones se desvían significativamente, se activan alertas y se evalúa reentrenamiento.


### 5.1 Estadísticas básicas

In [12]:
from pyspark.sql.functions import min as spark_min, max as spark_max, avg, stddev, count

stats = predictions_batch.select(
    spark_min("prediction").alias("min_pred"),
    spark_max("prediction").alias("max_pred"),
    avg("prediction").alias("avg_pred"),
    stddev("prediction").alias("std_pred"),
    count("*").alias("total")
).collect()[0]

print("Estadísticas de predicciones")
print(f"Total: {stats['total']:,}")
print(f"Mínimo: ${stats['min_pred']:,.2f}")
print(f"Máximo: ${stats['max_pred']:,.2f}")
print(f"Promedio: ${stats['avg_pred']:,.2f}")
print(f"Std: ${stats['std_pred']:,.2f}")


Estadísticas de predicciones
Total: 52,248
Mínimo: $-24,923,415,075.59
Máximo: $97,430,253,284.94
Promedio: $560,687,899.50
Std: $3,732,487,634.45


### 5.2 Distribución por rangos

In [13]:
from pyspark.sql.functions import when

prediction_ranges = predictions_batch.select(
    count(when(col("prediction") < 10_000_000, True)).alias("< 10M"),
    count(when((col("prediction") >= 10_000_000) & (col("prediction") < 100_000_000), True)).alias("10M-100M"),
    count(when((col("prediction") >= 100_000_000) & (col("prediction") < 1_000_000_000), True)).alias("100M-1B"),
    count(when(col("prediction") >= 1_000_000_000, True)).alias("> 1B")
)

prediction_ranges.show()


+-----+--------+-------+-----+
|< 10M|10M-100M|100M-1B| > 1B|
+-----+--------+-------+-----+
|24851|     126|  12885|14386|
+-----+--------+-------+-----+



### 5.3 Anomalias

In [14]:
anomalias = predictions_batch.filter(col("prediction") < 0).count()
print(f"Predicciones negativas: {anomalias}")


Predicciones negativas: 24840


## 6. Reto 5: Guardar resultados en Parquet y CSV

**Objetivo**: Almacenar predicciones en formatos consumibles.

**Instrucciones**:
1. Guarda en Parquet (óptimo para analytics y Spark)
2. Guarda en CSV (para consumo externo, Excel, etc.)
3. Verifica que los archivos se guardaron correctamente

**Pregunta**: ¿Qué formato usarías para cada caso?
- Dashboard interno: Parquet
- Reporte para gerencia: CSV
- Input para otro sistema: Parquet o API

In [15]:
predictions_output = "/opt/spark-data/processed/predictions_produccion"

predictions_batch.write.mode("overwrite") \
    .parquet(predictions_output + "/parquet")

predictions_batch.select(
    "prediction", "prediction_timestamp"
).write.mode("overwrite") \
 .option("header", "true") \
 .csv(predictions_output + "/csv")

print("Resultados guardados correctamente.")


Resultados guardados correctamente.


## 7. Reto 6: Diseñar pipeline de produccion automatizado

**Objetivo**: Pensar en cómo automatizar este proceso.

**Pregunta de diseño**: En un sistema real, este notebook se ejecutaría periódicamente. Diseña (en comentarios) cómo lo automatizarías:

1. **Frecuencia**: ¿Cada hora? ¿Cada día? ¿Bajo demanda?
2. **Orquestador**: ¿Airflow? ¿Cron? ¿Spark Streaming?
3. **Monitoreo**: ¿Cómo detectas si el modelo se degrada?
4. **Reentrenamiento**: ¿Cuándo reentrenar el modelo?
5. **Alertas**: ¿Qué condiciones disparan una alerta?

**Diseña tu pipeline de producción**

1. Frecuencia: Ejecución diaria o bajo demanda según volumen de contratos
2. Orquestador: Apache Airflow con DAG diario
3. Monitoreo: Drift estadístico + comparación de RMSE histórico. Monitoreado de manera diaria.
4. Reentrenamiento: Cuando el error supere un umbral o cada mes.
5. Alertas: Predicciones negativas, drift severo, caída de performance, errores seguidos o warnings. 

## 8. Bonus: Simulacion de scoring continuo por lotes

**Objetivo**: Simular el procesamiento de "lotes" de datos nuevos.

**Instrucciones**:
1. Divide los datos en 3 "lotes" simulados
2. Para cada lote, genera predicciones y calcula estadísticas
3. Compara estadísticas entre lotes
4. ¿Las predicciones son consistentes?

In [16]:
from pyspark.sql.functions import avg

batches = df_new_no_label.randomSplit([0.33, 0.33, 0.34], seed=42)

for i, batch in enumerate(batches):
    preds = production_model.transform(batch)
    avg_pred = preds.select(avg("prediction")).collect()[0][0]
    count_pred = preds.count()
    print(f"Lote {i+1}: {count_pred:,} registros | Promedio: ${avg_pred:,.2f}")


Lote 1: 17,280 registros | Promedio: $564,972,575.84


Lote 2: 17,399 registros | Promedio: $559,783,924.33


Lote 3: 17,569 registros | Promedio: $557,368,931.82


Las predicciones son consistentes entre lotes, lo que indica estabilidad del modelo en producción.

## 9. Preguntas de reflexión

**¿Qué pasa si los datos nuevos tienen un esquema diferente al de entrenamiento?**

*Respuesta:* El modelo fallaría o produciría resultados incorrectos. Se debe validar esquema antes de inferir.

**¿Cómo implementarías A/B testing en producción?**

*Respuesta:*  Usar dos versiones del modelo en paralelo y comparar métricas de negocio.

**¿Cuándo deberías retirar un modelo de producción?**

*Respuesta:* Cuando hay drift severo, caída sostenida de métricas o incumplimiento de objetivos.

**¿Qué métricas de monitoreo son más importantes para tu caso de uso?**

*Respuesta:* Distribución de predicciones, drift, latencia, volumen y errores.


In [18]:

print("Resumen inferencia en producción")

print("Verifica que hayas completado:")
print("  [✓] Modelo cargado desde MLflow Registry")
print("  [✓] Predicciones batch generadas")
print("  [✓] Estadísticas de predicciones calculadas")
print("  [✓] Resultados guardados (Parquet y CSV)")
print("  [✓] Pipeline de producción diseñado")
print("\nPróximos pasos sugeridos:")
print("  1. Configurar monitoreo de data drift")
print("  2. Implementar A/B testing")
print("  3. Automatizar reentrenamiento periódico")
print("  4. Crear alertas de anomalías en predicciones")
print("="*60)
print(" Diplomado: Finalizado:) ")


Resumen inferencia en producción
Verifica que hayas completado:
  [✓] Modelo cargado desde MLflow Registry
  [✓] Predicciones batch generadas
  [✓] Estadísticas de predicciones calculadas
  [✓] Resultados guardados (Parquet y CSV)
  [✓] Pipeline de producción diseñado

Próximos pasos sugeridos:
  1. Configurar monitoreo de data drift
  2. Implementar A/B testing
  3. Automatizar reentrenamiento periódico
  4. Crear alertas de anomalías en predicciones
 Diplomado: Finalizado:) 


In [19]:
spark.stop()
